In [1]:
import pandas as pd
from pandas import DataFrame
from common.utils import load_dataset, optimize_memory, get_params, DatasetType
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from common.math.functions import haversine

In [2]:
train_df: DataFrame = load_dataset("nyc-taxi-trip-duration", DatasetType.TRAIN, index=True)
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/home/jason/Desktop/data/nyc-taxi-trip-duration/train.csv'

In [ ]:
test_df: DataFrame = load_dataset("nyc-taxi-trip-duration", DatasetType.TEST, index=True)
train_df.head()

In [ ]:
import numpy as np

# Convert pickup_datetime to datetime
train_df['pickup_datetime'] = pd.to_datetime(train_df['pickup_datetime'])
test_df['pickup_datetime'] = pd.to_datetime(test_df['pickup_datetime'])

# Time-based features
for df in [train_df, test_df]:
    df['pickup_hour'] = df['pickup_datetime'].dt.hour
    df['pickup_day'] = df['pickup_datetime'].dt.dayofweek
    df['pickup_month'] = df['pickup_datetime'].dt.month

train_df['distance_km'] = haversine(train_df['pickup_latitude'], train_df['pickup_longitude'],
                                    train_df['dropoff_latitude'], train_df['dropoff_longitude'])

test_df['distance_km'] = haversine(test_df['pickup_latitude'], test_df['pickup_longitude'],
                                   test_df['dropoff_latitude'], test_df['dropoff_longitude'])

# Encode store_and_fwd_flag
train_df['store_and_fwd_flag'] = train_df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})
test_df['store_and_fwd_flag'] = test_df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    'vendor_id', 'passenger_count', 'pickup_hour', 'pickup_day',
    'pickup_month', 'store_and_fwd_flag', 'distance_km'
]

# Target variable (log-transform)
train_df['trip_duration'] = np.log1p(train_df['trip_duration'])

X = train_df[features]
y = train_df['trip_duration']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=20
)


In [ ]:
# Predict log-transformed duration
y_pred_log = model.predict(X_val)

# Compute RMSE in log space
rmse_log = np.sqrt(mean_squared_error(y_val, y_pred_log))

# Convert back to seconds
y_val_actual = np.expm1(y_val)
y_pred_actual = np.expm1(y_pred_log)
rmse_real = np.sqrt(mean_squared_error(y_val_actual, y_pred_actual))

print(f"RMSE (log-transformed): {rmse_log:.4f}")
print(f"RMSE (actual seconds): {rmse_real:.2f}")

In [ ]:
X_test = test_df[features]
test_preds = model.predict(X_test)
test_df['trip_duration'] = np.expm1(test_preds)

submission = test_df[['id', 'trip_duration']]
submission.to_csv('data/xgb_submission.csv', index=False)


In [ ]:
model.save_model("../../models/xgb_nyc_trip_duration.json")